<a href="https://colab.research.google.com/github/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/Hugging_Face_Trainer_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hugging Face Trainer API**

In [1]:
!pip install transformers datasets accelerate -q


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
import torch


# Um dataset *Supervised Fine-Tuning* (SFT)

In [3]:
data = [
    {"instruction": "O que é aprendizado de máquina?",
     "response": "Aprendizado de máquina é uma área da inteligência artificial que cria modelos capazes de aprender padrões a partir de dados."},

    {"instruction": "Explique o que é overfitting.",
     "response": "Overfitting ocorre quando um modelo aprende demais os dados de treino e não generaliza bem para novos dados."},

    {"instruction": "O que é uma árvore de decisão?",
     "response": "Uma árvore de decisão é um modelo supervisionado que usa regras em formato de árvore para realizar classificações ou regressões."}
]

dataset = Dataset.from_list(data)
dataset


Dataset({
    features: ['instruction', 'response'],
    num_rows: 3
})

# Load do Modelo e Tokenizer

In [6]:
model_name = "Qwen/Qwen2.5-0.5B"     # leve e roda no Colab
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  # se você tiver VRAM suficiente
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # garantir token de padding

model = AutoModelForCausalLM.from_pretrained(model_name)



config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

# Transformação para o formato de treino

LLMs do tipo causal exigem concatenar `input + resposta` como um único texto.

In [5]:
def tokenize(example):
    prompt = f"### Pergunta:\n{example['instruction']}\n\n### Resposta:\n{example['response']}"
    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=False)
tokenized_dataset


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})

# Configuração do Trainer

In [7]:
training_args = TrainingArguments(
    output_dir="./finetuned-model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=20,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),   # usa FP16 se tiver GPU
)


# Treino

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)


In [9]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rdeoliveirag (rdeoliveirag-universidade-presbiteriana-mackenzie) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
1,14.408800
2,14.408800
3,14.408800


TrainOutput(global_step=3, training_loss=14.408767700195312, metrics={'train_runtime': 141.663, 'train_samples_per_second': 0.064, 'train_steps_per_second': 0.021, 'total_flos': 4947583500288.0, 'train_loss': 14.408767700195312, 'epoch': 3.0})

# Chamada do Modelo

In [11]:
def gerar_texto(pergunta):
    prompt = f"### Pergunta:\n{pergunta}\n\n### Resposta:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=128)
    print(tokenizer.decode(output[0], skip_special_tokens=True))

gerar_texto("Explique o conceito de regressão linear.")
gerar_texto("Explique o conceito de overfitting.")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


### Pergunta:
Explique o conceito de regressão linear.

### Resposta:
A regressão linear é uma técnica de predição utilizada em ciências da computação e de dados para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as linhas de regressão para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as linhas de regressão para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as lin
### Pergunta:
Explique o conceito de overfitting.

### Resposta:
Overfitting é um problema de aprendizado de máquina em que o modelo é muito bem ajustado a um conjunto de dados, mas não é capaz de prever o comportamento de dados que não foram usados para treinar o modelo. Isso ocorre quando o modelo é muito complexo e não tem suficiente informação para prever o c

# LoRA

## Install & import `peft` (`LoRA`)

In [12]:
!pip install peft -q


In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import torch

## Faz as Mudanças do Modelo Carregado com o `LoRA`

In [14]:
lora_config = LoraConfig(
    r=16,                       # rank do LoRA
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # só para verificar


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## Treino

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)


In [16]:
trainer.train()


Step,Training Loss
1,14.408800
2,14.408800
3,14.408800


TrainOutput(global_step=3, training_loss=14.408767700195312, metrics={'train_runtime': 2.1943, 'train_samples_per_second': 4.101, 'train_steps_per_second': 1.367, 'total_flos': 4962531999744.0, 'train_loss': 14.408767700195312, 'epoch': 3.0})

## Chamada do Modelo

In [17]:
def gerar_texto(pergunta):
    prompt = f"### Pergunta:\n{pergunta}\n\n### Resposta:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=128)
    print(tokenizer.decode(output[0], skip_special_tokens=True))

gerar_texto("Explique o conceito de regressão linear.")
gerar_texto("Explique o conceito de overfitting.")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


### Pergunta:
Explique o conceito de regressão linear.

### Resposta:
A regressão linear é uma técnica de predição utilizada em ciências da computação e de dados para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as linhas de regressão para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as linhas de regressão para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as lin
### Pergunta:
Explique o conceito de overfitting.

### Resposta:
Overfitting é um problema de aprendizado de máquina em que o modelo é muito bem ajustado a um conjunto de dados, mas não é capaz de prever o comportamento de dados que não foram usados para treinar o modelo. Isso ocorre quando o modelo é muito complexo e não tem suficiente informação para prever o c